In [1]:

# ============================================================================
# KÜTÜPHANE YÜKLEME
# ============================================================================

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import datetime

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Model Kütüphaneleri
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# Diğer
import joblib
import json

warnings.filterwarnings('ignore')

print("✅ Tüm kütüphaneler yüklendi")

# ============================================================================
# AYARLAR
# ============================================================================

# Dataset yolu
DATA_PATH = r"C:\Users\seray\OneDrive\Masaüstü\data\BenignAndMaliciousDataset.csv"

# Output klasörü
OUTPUT_DIR = Path("./model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Random seed (tekrarlanabilirlik için)
RANDOM_STATE = 42

# Test split oranı
TEST_SIZE = 0.2

# Model isimleri (yeni modeller için)
MODEL_VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
NEW_MODEL_NAME = f"chatsecops_model_v2_{MODEL_VERSION}"

print(f"✅ Ayarlar yapılandırıldı")
print(f"   Dataset: {DATA_PATH}")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Model adı: {NEW_MODEL_NAME}")
print(f"   Random state: {RANDOM_STATE}")
print(f"   Test size: {TEST_SIZE * 100}%")

# ============================================================================
# YARDIMCI FONKSİYONLAR
# ============================================================================

def print_section(title):
    """Başlık yazdır"""
    print("\n" + "="*70)
    print(f" {title}")
    print("="*70 + "\n")

def save_metadata(metadata, filename="metadata.json"):
    """Metadata kaydet"""
    filepath = OUTPUT_DIR / filename
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    print(f"💾 Metadata kaydedildi: {filepath}")
    return filepath

print("✅ Yardımcı fonksiyonlar tanımlandı")
print("\n🎉 HAZIRLIK TAMAMLANDI - Cell 2'ye geçebilirsiniz!")

✅ Tüm kütüphaneler yüklendi
✅ Ayarlar yapılandırıldı
   Dataset: C:\Users\seray\OneDrive\Masaüstü\data\BenignAndMaliciousDataset.csv
   Output: model_outputs
   Model adı: chatsecops_model_v2_20260114_203833
   Random state: 42
   Test size: 20.0%
✅ Yardımcı fonksiyonlar tanımlandı

🎉 HAZIRLIK TAMAMLANDI - Cell 2'ye geçebilirsiniz!


In [2]:
"""
ChatSecOps Model Development - Cell 2
Veri Yükleme ve İlk İnceleme
"""

print_section("VERİ YÜKLEME VE İLK İNCELEME")

# ============================================================================
# VERİ YÜKLEME
# ============================================================================

try:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Dataset başarıyla yüklendi")
    print(f"   Boyut: {df.shape[0]:,} satır x {df.shape[1]} sütun")
except FileNotFoundError:
    print(f"❌ HATA: Dosya bulunamadı: {DATA_PATH}")
    print("   Lütfen dosya yolunu kontrol edin!")
    raise
except Exception as e:
    print(f"❌ Beklenmeyen hata: {e}")
    raise

# ============================================================================
# İLK İNCELEME
# ============================================================================

print("\n📊 İlk 5 satır:")
print(df.head())

print("\n📋 Sütun bilgileri:")
print(df.info())

print("\n🔍 Temel istatistikler:")
print(df.describe())

# ============================================================================
# HEDEF DEĞİŞKEN ANALİZİ
# ============================================================================

print("\n🎯 Hedef değişken (Class) dağılımı:")
class_counts = df['Class'].value_counts()
print(class_counts)

class_percentages = df['Class'].value_counts(normalize=True) * 100
print(f"\nOranlar:")
print(f"  Class 0 (Benign):    {class_counts[0]:,} ({class_percentages[0]:.2f}%)")
print(f"  Class 1 (Malicious): {class_counts[1]:,} ({class_percentages[1]:.2f}%)")

# Dengesiz veri kontrolü
if abs(class_percentages[0] - class_percentages[1]) > 20:
    print("\n⚠️  DİKKAT: Veri dengesiz! Stratified sampling kullanacağız.")
else:
    print("\n✅ Veri dengeli.")

# ============================================================================
# KAYIP VERİ ANALİZİ
# ============================================================================

print("\n🔍 Kayıp veri analizi:")
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if len(missing_data) > 0:
    print("   Kayıp veri olan sütunlar:")
    for col, count in missing_data.items():
        percentage = (count / len(df)) * 100
        print(f"   • {col}: {count:,} ({percentage:.2f}%)")
else:
    print("   ✅ Kayıp veri yok!")

# ============================================================================
# SÜTUN TİPLERİ
# ============================================================================

print("\n📑 Sütun tipleri:")
print(f"   Numerik: {len(df.select_dtypes(include=['int64', 'float64']).columns)}")
print(f"   Kategorik: {len(df.select_dtypes(include=['object']).columns)}")
print(f"   Boolean: {len(df.select_dtypes(include=['bool']).columns)}")

print("\n✅ VERİ YÜKLEME TAMAMLANDI - Cell 3'e geçebilirsiniz!")


 VERİ YÜKLEME VE İLK İNCELEME

✅ Dataset başarıyla yüklendi
   Boyut: 90,000 satır x 34 sütun

📊 İlk 5 satır:
   Domain DNSRecordType  MXDnsResponse  TXTDnsResponse  HasSPFInfo  \
0    4455             A          False           False       False   
1    4456             A          False           False       False   
2    4457             A          False           False       False   
3    4458             A          False           False       False   
4    4459             A          False           False       False   

   HasDkimInfo  HasDmarcInfo     Ip  DomainInAlexaDB  CommonPorts  ...  \
0        False         False  16984            False        False  ...   
1        False         False  16984            False        False  ...   
2        False         False  16984            False        False  ...   
3        False         False  16984            False        False  ...   
4        False         False  16984            False        False  ...   

  ConsoantRatio Numeric

In [3]:
"""
ChatSecOps Model Development - Cell 3
Veri Ön İşleme (Preprocessing)
"""

print_section("VERİ ÖN İŞLEME")

# ============================================================================
# 1. KAYIP VERİLERİ DOLDURMA
# ============================================================================

print("🔧 Adım 1: Kayıp verileri doldurma...")

# Object (metin) sütunları "Bilinmiyor" ile doldur
object_columns = df.select_dtypes(include=['object']).columns
for col in object_columns:
    df[col] = df[col].fillna("Bilinmiyor")

print(f"   ✅ {len(object_columns)} metin sütunundaki kayıp veriler dolduruldu")

# Numerik sütunlarda kayıp veri varsa median ile doldur
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
numeric_columns = [col for col in numeric_columns if col != 'Class']

for col in numeric_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f"   ✅ {col}: median ile dolduruldu")

# ============================================================================
# 2. YÜKSEK KARDİNALİTE SÜTUNLARINI ÇIKARMA
# ============================================================================

print("\n🔧 Adım 2: Yüksek kardinalite sütunları çıkarma...")

# RegisteredOrg ve Domain sütunlarını çıkar
columns_to_drop = ['RegisteredOrg', 'Domain']
df_processed = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print(f"   ✅ Çıkarılan sütunlar: {[col for col in columns_to_drop if col in df.columns]}")

# ============================================================================
# 3. TLD GRUPLAŞTIRMA
# ============================================================================

print("\n🔧 Adım 3: TLD gruplaştırma...")

if 'TLD' in df_processed.columns:
    # En sık kullanılan 30 TLD'yi bul
    top_30_tlds = df_processed['TLD'].value_counts().nlargest(30).index.tolist()
    
    # Diğerlerini 'TLD_Other' olarak grupla
    df_processed['TLD_Grouped'] = df_processed['TLD'].apply(
        lambda x: x if x in top_30_tlds else 'TLD_Other'
    )
    
    # Orijinal TLD sütununu çıkar
    df_processed = df_processed.drop(columns=['TLD'])
    
    print(f"   ✅ TLD gruplaştırıldı (Top 30 + Other)")
    print(f"   Top 5 TLD: {top_30_tlds[:5]}")
else:
    top_30_tlds = []
    print("   ⚠️  TLD sütunu bulunamadı, atlanıyor...")

# ============================================================================
# 4. ONE-HOT ENCODING
# ============================================================================

print("\n🔧 Adım 4: One-Hot Encoding...")

# OHE uygulanacak sütunlar
ohe_columns = ['DNSRecordType', 'CountryCode', 'RegisteredCountry', 'TLD_Grouped']
ohe_columns = [col for col in ohe_columns if col in df_processed.columns]

print(f"   OHE uygulanacak sütunlar: {ohe_columns}")

# OHE uygula
df_processed = pd.get_dummies(df_processed, columns=ohe_columns, dtype=int)

print(f"   ✅ One-Hot Encoding tamamlandı")
print(f"   Yeni sütun sayısı: {df_processed.shape[1]}")

# ============================================================================
# 5. ÖZELLİKLER VE HEDEF DEĞİŞKENİ AYIRMA
# ============================================================================

print("\n🔧 Adım 5: Özellikler ve hedef ayrılıyor...")

# Hedef değişken (y)
y = df_processed['Class']

# Özellikler (X)
X = df_processed.drop(columns=['Class'])

print(f"   ✅ X (Features): {X.shape}")
print(f"   ✅ y (Target): {y.shape}")

# Boolean sütunları int'e çevir
bool_columns = X.select_dtypes(include=['bool']).columns
for col in bool_columns:
    X[col] = X[col].astype(int)

print(f"   ✅ {len(bool_columns)} boolean sütun int'e çevrildi")

# ============================================================================
# 6. ÖLÇEKLENDİRME (SCALING)
# ============================================================================

print("\n🔧 Adım 6: Numerik özellikleri ölçeklendirme...")

# Ölçeklendirilecek sütunları belirle
float_columns = X.select_dtypes(include=['float64']).columns
int_columns = [col for col in X.select_dtypes(include=['int64']).columns if X[col].max() > 1]
columns_to_scale = list(float_columns) + list(int_columns)

print(f"   Ölçeklendirilecek {len(columns_to_scale)} sütun bulundu")

# Scaler oluştur ve uygula
scaler = StandardScaler()
X[columns_to_scale] = scaler.fit_transform(X[columns_to_scale])

print(f"   ✅ Ölçeklendirme tamamlandı")

# ============================================================================
# 7. TRAIN-TEST SPLIT
# ============================================================================

print("\n🔧 Adım 7: Train-Test split...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y  # Sınıf dağılımını koru
)

print(f"   ✅ Veri ayrıldı:")
print(f"      Training: {X_train.shape[0]:,} samples")
print(f"      Testing: {X_test.shape[0]:,} samples")
print(f"      Features: {X_train.shape[1]}")

# ============================================================================
# METADATA KAYDET
# ============================================================================

preprocessing_metadata = {
    'timestamp': datetime.now().isoformat(),
    'original_shape': df.shape,
    'processed_shape': X.shape,
    'n_features': X.shape[1],
    'feature_names': X.columns.tolist(),
    'columns_to_scale': columns_to_scale,
    'top_30_tlds': top_30_tlds,
    'ohe_columns': ohe_columns,
    'train_size': X_train.shape[0],
    'test_size': X_test.shape[0]
}

save_metadata(preprocessing_metadata, f"preprocessing_metadata_{MODEL_VERSION}.json")

print("\n✅ VERİ ÖN İŞLEME TAMAMLANDI - Cell 4'e geçebilirsiniz!")
print(f"\n📊 Özet:")
print(f"   Orijinal: {df.shape[0]:,} satır x {df.shape[1]} sütun")
print(f"   İşlenmiş: {X.shape[0]:,} satır x {X.shape[1]} sütun")
print(f"   Training: {X_train.shape[0]:,} satır")
print(f"   Testing: {X_test.shape[0]:,} satır")


 VERİ ÖN İŞLEME

🔧 Adım 1: Kayıp verileri doldurma...
   ✅ 5 metin sütunundaki kayıp veriler dolduruldu

🔧 Adım 2: Yüksek kardinalite sütunları çıkarma...
   ✅ Çıkarılan sütunlar: ['RegisteredOrg', 'Domain']

🔧 Adım 3: TLD gruplaştırma...
   ✅ TLD gruplaştırıldı (Top 30 + Other)
   Top 5 TLD: ['com', 'net', 'online', 'com.br', 'org']

🔧 Adım 4: One-Hot Encoding...
   OHE uygulanacak sütunlar: ['DNSRecordType', 'CountryCode', 'RegisteredCountry', 'TLD_Grouped']
   ✅ One-Hot Encoding tamamlandı
   Yeni sütun sayısı: 285

🔧 Adım 5: Özellikler ve hedef ayrılıyor...
   ✅ X (Features): (90000, 284)
   ✅ y (Target): (90000,)
   ✅ 9 boolean sütun int'e çevrildi

🔧 Adım 6: Numerik özellikleri ölçeklendirme...
   Ölçeklendirilecek 18 sütun bulundu
   ✅ Ölçeklendirme tamamlandı

🔧 Adım 7: Train-Test split...
   ✅ Veri ayrıldı:
      Training: 72,000 samples
      Testing: 18,000 samples
      Features: 284
💾 Metadata kaydedildi: model_outputs\preprocessing_metadata_20260114_203833.json

✅ VERİ Ö

In [4]:
"""
ChatSecOps Model Development - Cell 4
Model Eğitimi - Tüm Modelleri Test Et
"""

print_section("MODEL EĞİTİMİ - KARŞILAŞTIRMA")

import time

# ============================================================================
# MODEL TANIMLAMA
# ============================================================================

print("🤖 Eğitilecek modeller:")

models_to_train = {
    'Logistic_Regression': LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'Random_Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_depth=20
    ),
    'Gradient_Boosting': GradientBoostingClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        max_depth=10
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        verbose=-1,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        verbosity=0,
        n_jobs=-1,
        eval_metric='logloss'
    )
}

for i, model_name in enumerate(models_to_train.keys(), 1):
    print(f"   {i}. {model_name}")

print(f"\nToplam {len(models_to_train)} model eğitilecek")

# ============================================================================
# EĞİTİM VE DEĞERLENDİRME
# ============================================================================

results = {}
trained_models = {}

print("\n" + "="*70)

for model_name, model in models_to_train.items():
    print(f"\n🔧 {model_name} eğitiliyor...")
    
    # Eğitim süresi ölç
    start_time = time.time()
    
    # Modeli eğit
    model.fit(X_train, y_train)
    
    training_time = time.time() - start_time
    
    # Tahmin yap
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrikleri hesapla
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba),
        'training_time_seconds': training_time
    }
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    metrics.update({
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp),
        'false_positive_rate': fp / (fp + tn),
        'false_negative_rate': fn / (fn + tp)
    })
    
    # Sonuçları kaydet
    results[model_name] = metrics
    trained_models[model_name] = model
    
    # Özet yazdır
    print(f"   ✅ Eğitim tamamlandı ({training_time:.2f} saniye)")
    print(f"   • Accuracy:  {metrics['accuracy']*100:.2f}%")
    print(f"   • Precision: {metrics['precision']*100:.2f}%")
    print(f"   • Recall:    {metrics['recall']*100:.2f}%")
    print(f"   • F1-Score:  {metrics['f1_score']*100:.2f}%")
    print(f"   • ROC-AUC:   {metrics['roc_auc']:.4f}")

print("\n" + "="*70)
print("\n✅ TÜM MODELLERİN EĞİTİMİ TAMAMLANDI!")


 MODEL EĞİTİMİ - KARŞILAŞTIRMA

🤖 Eğitilecek modeller:
   1. Logistic_Regression
   2. Random_Forest
   3. Gradient_Boosting
   4. LightGBM
   5. XGBoost

Toplam 5 model eğitilecek


🔧 Logistic_Regression eğitiliyor...
   ✅ Eğitim tamamlandı (4.49 saniye)
   • Accuracy:  99.38%
   • Precision: 99.70%
   • Recall:    99.06%
   • F1-Score:  99.38%
   • ROC-AUC:   0.9996

🔧 Random_Forest eğitiliyor...
   ✅ Eğitim tamamlandı (1.75 saniye)
   • Accuracy:  99.61%
   • Precision: 99.88%
   • Recall:    99.33%
   • F1-Score:  99.60%
   • ROC-AUC:   1.0000

🔧 Gradient_Boosting eğitiliyor...
   ✅ Eğitim tamamlandı (64.71 saniye)
   • Accuracy:  99.77%
   • Precision: 99.87%
   • Recall:    99.68%
   • F1-Score:  99.77%
   • ROC-AUC:   1.0000

🔧 LightGBM eğitiliyor...
   ✅ Eğitim tamamlandı (0.69 saniye)
   • Accuracy:  99.75%
   • Precision: 99.82%
   • Recall:    99.68%
   • F1-Score:  99.75%
   • ROC-AUC:   1.0000

🔧 XGBoost eğitiliyor...
   ✅ Eğitim tamamlandı (3.62 saniye)
   • Accuracy:  9

In [8]:
"""
ChatSecOps Model Development - Cell 6
Model ve Metadata Kaydetme
"""

# ============================================================================
# 0. MODEL SEÇİMİ (Manuel olarak LightGBM seçildi)
# ============================================================================
best_model_name = 'LightGBM' 
best_model = trained_models[best_model_name]

print(f"🚀 Hız Şampiyonu Seçildi: {best_model_name}")
print(f"🎯 Accuracy: {results[best_model_name]['accuracy']*100:.2f}%")
print(f"⚡ Eğitim Süresi: {results[best_model_name]['training_time_seconds']:.2f} saniye")

# Karşılaştırma tablosunu rapor için hazırla
comparison_display = pd.DataFrame(results).T

print_section("MODEL VE METADATA KAYDETME")

# ============================================================================
# 1. EN İYİ MODELİ KAYDET
# ============================================================================

print("💾 En iyi model kaydediliyor...")

model_filename = f"{NEW_MODEL_NAME}.joblib"
model_path = OUTPUT_DIR / model_filename

joblib.dump(best_model, model_path)
print(f"   ✅ Model kaydedildi: {model_path}")

# ============================================================================
# 2. SCALER'I KAYDET
# ============================================================================

print("\n💾 Scaler kaydediliyor...")

scaler_filename = f"{NEW_MODEL_NAME}_scaler.joblib"
scaler_path = OUTPUT_DIR / scaler_filename

joblib.dump(scaler, scaler_path)
print(f"   ✅ Scaler kaydedildi: {scaler_path}")

# ============================================================================
# 3. KAPSAMLI METADATA OLUŞTUR
# ============================================================================

print("\n💾 Metadata oluşturuluyor...")

# Test seti tahminleri
y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)

# Detaylı classification report
class_report = classification_report(y_test, y_pred_best, output_dict=True)

# Metadata paketi
complete_metadata = {
    # Model Bilgileri
    'model_info': {
        'model_name': best_model_name,
        'model_type': type(best_model).__name__,
        'version': MODEL_VERSION,
        'created_at': datetime.now().isoformat(),
        'random_state': RANDOM_STATE
    },
    
    # Dataset Bilgileri
    'dataset_info': {
        'source': str(DATA_PATH),
        'total_samples': int(len(df)),
        'n_features': int(X.shape[1]),
        'feature_names': X.columns.tolist(),
        'train_size': int(len(X_train)),
        'test_size': int(len(X_test)),
        'class_distribution': {
            'benign': int((y == 0).sum()),
            'malicious': int((y == 1).sum())
        }
    },
    
    # Preprocessing Bilgileri
    'preprocessing': {
        'columns_to_scale': columns_to_scale,
        'scaled_features_count': len(columns_to_scale),
        'top_30_tlds': top_30_tlds,
        'ohe_applied': True,
        'missing_values_handled': True
    },
    
    # Performans Metrikleri
    'performance': {
        'test_metrics': results[best_model_name],
        'classification_report': class_report
    },
    
    # Model Comparison
    'all_models_comparison': {
        model: {
            'accuracy': float(metrics['accuracy']),
            'f1_score': float(metrics['f1_score']),
            'roc_auc': float(metrics['roc_auc'])
        }
        for model, metrics in results.items()
    },
    
    # Dosya Yolları
    'files': {
        'model_file': str(model_path),
        'scaler_file': str(scaler_path),
        'model_filename': model_filename,
        'scaler_filename': scaler_filename
    },
    
    # Production Hazırlık
    'production_ready': {
        'accuracy_threshold': 0.99,
        'meets_accuracy': float(results[best_model_name]['accuracy']) >= 0.99,
        'false_positive_rate': float(results[best_model_name]['false_positive_rate']),
        'false_negative_rate': float(results[best_model_name]['false_negative_rate']),
        'recommended_for_production': (
            float(results[best_model_name]['accuracy']) >= 0.99 and
            float(results[best_model_name]['false_positive_rate']) < 0.01 and
            float(results[best_model_name]['false_negative_rate']) < 0.005
        )
    }
}

# Metadata kaydet
metadata_filename = f"{NEW_MODEL_NAME}_metadata.json"
metadata_path = OUTPUT_DIR / metadata_filename

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(complete_metadata, f, indent=2, ensure_ascii=False)

print(f"   ✅ Metadata kaydedildi: {metadata_path}")

# ============================================================================
# 4. HUMAN-READABLE RAPOR OLUŞTUR
# ============================================================================

print("\n📄 Human-readable rapor oluşturuluyor...")

report_filename = f"{NEW_MODEL_NAME}_REPORT.txt"
report_path = OUTPUT_DIR / report_filename

with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*70 + "\n")
    f.write("ChatSecOps Model Training Report\n")
    f.write("="*70 + "\n\n")
    
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model Version: {MODEL_VERSION}\n\n")
    
    f.write("="*70 + "\n")
    f.write("BEST MODEL\n")
    f.write("="*70 + "\n\n")
    f.write(f"Model: {best_model_name}\n")
    f.write(f"Accuracy: {results[best_model_name]['accuracy']*100:.2f}%\n")
    f.write(f"F1-Score: {results[best_model_name]['f1_score']:.4f}\n")
    f.write(f"ROC-AUC: {results[best_model_name]['roc_auc']:.4f}\n\n")
    
    f.write("="*70 + "\n")
    f.write("ALL MODELS COMPARISON\n")
    f.write("="*70 + "\n\n")
    f.write(comparison_display.to_string())
    f.write("\n\n")
    
    f.write("="*70 + "\n")
    f.write("PRODUCTION READINESS\n")
    f.write("="*70 + "\n\n")
    
    prod_ready = complete_metadata['production_ready']
    f.write(f"Recommended for Production: {'YES' if prod_ready['recommended_for_production'] else 'NO'}\n\n")
    f.write(f"Accuracy: {prod_ready['meets_accuracy']} (Threshold: {prod_ready['accuracy_threshold']})\n")
    f.write(f"False Positive Rate: {prod_ready['false_positive_rate']*100:.2f}% (Target: <1%)\n")
    f.write(f"False Negative Rate: {prod_ready['false_negative_rate']*100:.2f}% (Target: <0.5%)\n\n")
    
    f.write("="*70 + "\n")
    f.write("FILES\n")
    f.write("="*70 + "\n\n")
    f.write(f"Model: {model_filename}\n")
    f.write(f"Scaler: {scaler_filename}\n")
    f.write(f"Metadata: {metadata_filename}\n")

print(f"   ✅ Rapor kaydedildi: {report_path}")

# ============================================================================
# ÖZET
# ============================================================================

print("\n" + "="*70)
print("🎉 TÜM DOSYALAR BAŞARIYLA KAYDEDİLDİ!")
print("="*70)

print(f"\n📦 Kaydedilen dosyalar ({OUTPUT_DIR}):")
print(f"   1. {model_filename}")
print(f"   2. {scaler_filename}")
print(f"   3. {metadata_filename}")
print(f"   4. {report_filename}")

print("\n💡 Sonraki adımlar:")
print("   1. REPORT.txt dosyasını okuyun")
print("   2. soar_engine.py'de model yollarını güncelleyin:")
print(f"      MODEL_PATH = 'model_outputs/{model_filename}'")
print(f"      SCALER_PATH = 'model_outputs/{scaler_filename}'")
print("   3. Test edin!")

print("\n✅ CELL 6 TAMAMLANDI - İsteğe bağlı Cell 7'ye geçebilirsiniz (Advanced Tests)")

🚀 Hız Şampiyonu Seçildi: LightGBM
🎯 Accuracy: 99.75%
⚡ Eğitim Süresi: 0.69 saniye

 MODEL VE METADATA KAYDETME

💾 En iyi model kaydediliyor...
   ✅ Model kaydedildi: model_outputs\chatsecops_model_v2_20260114_203833.joblib

💾 Scaler kaydediliyor...
   ✅ Scaler kaydedildi: model_outputs\chatsecops_model_v2_20260114_203833_scaler.joblib

💾 Metadata oluşturuluyor...
   ✅ Metadata kaydedildi: model_outputs\chatsecops_model_v2_20260114_203833_metadata.json

📄 Human-readable rapor oluşturuluyor...
   ✅ Rapor kaydedildi: model_outputs\chatsecops_model_v2_20260114_203833_REPORT.txt

🎉 TÜM DOSYALAR BAŞARIYLA KAYDEDİLDİ!

📦 Kaydedilen dosyalar (model_outputs):
   1. chatsecops_model_v2_20260114_203833.joblib
   2. chatsecops_model_v2_20260114_203833_scaler.joblib
   3. chatsecops_model_v2_20260114_203833_metadata.json
   4. chatsecops_model_v2_20260114_203833_REPORT.txt

💡 Sonraki adımlar:
   1. REPORT.txt dosyasını okuyun
   2. soar_engine.py'de model yollarını güncelleyin:
      MODEL_PATH = '